# Retail Data Wrangling and Analytics

## Data Preperation

In [0]:
from pyspark.sql import SparkSession

In [0]:
# load data
df = spark.table("LGS")
display(df.head(5))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00Z,6.75,13085.0,United Kingdom
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00Z,1.25,13085.0,United Kingdom


In [0]:
# convert column names into snakecase
df = df.withColumnRenamed("Invoice", "invoice_no") \
  .withColumnRenamed("StockCode", "stock_code") \
  .withColumnRenamed("Description", "description") \
  .withColumnRenamed("Quantity", "quantity") \
  .withColumnRenamed("InvoiceDate", "invoice_date") \
  .withColumnRenamed("Price", "unit_price") \
  .withColumnRenamed("Customer ID", "customer_id") \
  .withColumnRenamed("Country", "country")
display(df.head(5))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00Z,6.75,13085.0,United Kingdom
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00Z,1.25,13085.0,United Kingdom


In [0]:
# check current data types
display(df.dtypes)

_1,_2
invoice_no,string
stock_code,string
description,string
quantity,bigint
invoice_date,timestamp
unit_price,double
customer_id,double
country,string


In [0]:
# cast columns into appropriate types
from pyspark.sql.functions import col

retail_df = df.withColumn('customer_id', col('customer_id').cast('int'))
retail_df = retail_df.withColumn('invoice_date', col('invoice_date').cast('date'))

display(retail_df.dtypes)

_1,_2
invoice_no,string
stock_code,string
description,string
quantity,bigint
invoice_date,date
unit_price,double
customer_id,int
country,string


In [0]:
display(retail_df.head(5))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01,6.95,13085,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01,6.75,13085,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01,6.75,13085,United Kingdom
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.1,13085,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085,United Kingdom


## Invoice Amount Distribution

In [0]:
# calculate invoice amount
from pyspark.sql.functions import sum 

retail_df = retail_df.withColumn('invoice_amount', retail_df['quantity'] * retail_df['unit_price'])
inv_amt = retail_df.groupBy('invoice_no').agg(sum('invoice_amount').alias('invoice_amount'))

In [0]:
# filter out first 85 percentile and negatives
from pyspark.sql.functions import percentile_approx

quantile_85 = inv_amt.agg(percentile_approx('invoice_amount', 0.85)).collect()[0][0]
inv_amt = inv_amt.filter(
    (col('invoice_amount') >= 0) & (col('invoice_amount') <= quantile_85)
)

display(inv_amt.head(5))

invoice_no,invoice_amount
489677,192.0
491045,303.2
491658,155.05999999999997
493542,118.75
493977,275.95


## Monthly Placed and Canceled Order

In [0]:
# reformat invoice date in new column for easier group by
from pyspark.sql.functions import date_format

retail_df = retail_df.withColumn('inv_yr_month', date_format('invoice_date', 'yyyyMM').cast('int'))
display(retail_df.head(5))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,invoice_amount,inv_yr_month
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01,6.95,13085,United Kingdom,83.4,200912
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01,6.75,13085,United Kingdom,81.0,200912
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01,6.75,13085,United Kingdom,81.0,200912
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.1,13085,United Kingdom,100.80000000000001,200912
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085,United Kingdom,30.0,200912


In [0]:
from pyspark.sql.functions import countDistinct

# total orders by month
ttl_order = retail_df.groupBy('inv_yr_month').agg(
    countDistinct('invoice_no').alias('total_orders')
)

# cancelled orders per month
cncl_df = retail_df.filter(col('invoice_no').startswith('C'))
cncl_order = cncl_df.groupBy('inv_yr_month').agg(
    countDistinct('invoice_no').alias('cancelled_orders')
)


In [0]:
# combine and calculate placed orders per month
monthly_stat = ttl_order.join(cncl_order, on='inv_yr_month', how='left') \
    .fillna(0) \
    .withColumn('placed_orders', col('total_orders') - 2 * col('cancelled_orders')) \
    .select('inv_yr_month', 'placed_orders', 'cancelled_orders') \
    .orderBy('inv_yr_month')

display(monthly_stat.head(10))

inv_yr_month,placed_orders,cancelled_orders
200912,1528,401
201001,1033,300
201002,1489,240
201003,1553,407
201004,1284,304
201005,1604,407
201006,1502,357
201007,1329,344
201008,1331,273
201009,1633,371


## Monthly Sales

In [0]:
monthly_sales = retail_df.groupBy('inv_yr_month').agg(
    sum('invoice_amount').alias('sales')
)

display(monthly_sales.head(10))

inv_yr_month,sales
201011,1422654.6419998251
201101,560000.2600000234
201004,590580.4319999823
201003,765848.7609999765
201103,683267.0800000189
201012,1126445.4699999166
201001,624032.8919999956
201005,615322.8300000005
200912,799847.1100000143
201009,853650.4309999745


## Monthly Sales Growth

In [0]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

# define window
w = Window.orderBy('inv_yr_month')

# calculate percentage change
sales_growth = monthly_sales.withColumn('prev_sales', lag('sales', 1).over(w)) \
    .withColumn('pct_growth', ((col('sales') - col('prev_sales')) / col('prev_sales') * 100)) \
    .fillna(0).select('inv_yr_month', 'sales', 'pct_growth').orderBy('inv_yr_month')

display(sales_growth.head(10))

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


inv_yr_month,sales,pct_growth
200912,799847.1100000143,0.0
201001,624032.8919999956,-21.980978089677112
201002,533091.4260000042,-14.573184709627782
201003,765848.7609999765,43.661804269943474
201004,590580.4319999823,-22.885501410375667
201005,615322.8300000005,4.18950521544184
201006,679786.6099999842,10.476416095268814
201007,575236.359999999,-15.379863101449958
201008,656776.3399999854,14.17503928298039
201009,853650.4309999745,29.975819622246664


## Monthly Active Users

In [0]:
active = retail_df.groupBy('inv_yr_month').agg(countDistinct('customer_id').alias('active_users'))
display(active.head(10))

inv_yr_month,active_users
201108,980
201011,1683
201101,783
201004,998
201003,1111
201103,1020
201112,686
201012,948
201001,786
201005,1062


## New and Existing Users

In [0]:
# identify month of first purchase for each customer
from pyspark.sql.functions import min

first_pchs = retail_df.groupBy('customer_id').agg(min('inv_yr_month')).withColumnRenamed('min(inv_yr_month)', 'first_pchs')

display(first_pchs.head(10))

customer_id,first_pchs
13623,200912
17679,200912
17389,200912
18051,200912
13289,200912
17753,201001
15727,201001
16574,201002
14832,201003
15447,201003


In [0]:
# join the first purchase info with transactional data

txn = retail_df.join(first_pchs, on='customer_id', how='left')
display(txn.head(5))

customer_id,invoice_no,stock_code,description,quantity,invoice_date,unit_price,country,invoice_amount,inv_yr_month,first_pchs
13085,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01,6.95,United Kingdom,83.4,200912,200912
13085,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01,6.75,United Kingdom,81.0,200912,200912
13085,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01,6.75,United Kingdom,81.0,200912,200912
13085,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.1,United Kingdom,100.80000000000001,200912,200912
13085,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,United Kingdom,30.0,200912,200912


In [0]:
# label customer type

from pyspark.sql.functions import when

txn = txn.withColumn('customer_type', when(col('first_pchs') == col('inv_yr_month'), 'new').otherwise('existing'))

In [0]:
# monthly customer count based on labelled type

new_user = txn.filter(col('customer_type') == 'new').groupBy('inv_yr_month').agg(countDistinct('customer_id').alias('new'))
existing_user = txn.filter(col('customer_type') == 'existing').groupBy('inv_yr_month').agg(countDistinct('customer_id').alias('existing'))

user_type = new_user.join(existing_user, on='inv_yr_month').fillna(0)
display(user_type.head(10))

inv_yr_month,new,existing
201108,106,874
201011,322,1361
201101,71,712
201004,291,707
201003,436,675
201103,178,842
201112,28,658
201012,77,871
201001,394,392
201005,254,808


## RFM Segmentation

In [0]:
# we'll use the day right after the latest transaction date as "today"

from pyspark.sql.functions import *
from datetime import timedelta

tday = txn.agg(max('invoice_date')).collect()[0][0] + timedelta(days=1)

print(tday)

2011-12-10


In [0]:
# rfm table
rfm = txn.groupBy('customer_id').agg(
    datediff(lit(tday), max('invoice_date')).alias('recency'),
    countDistinct('invoice_no').alias('frequency'),
    sum('invoice_amount').alias('monetary')
)

display(rfm.head(5))

customer_id,recency,frequency,monetary
13623,31,15,2446.3599999999997
17679,53,11,3166.5600000000004
17389,1,77,54587.02999999999
18051,635,8,2275.98
13289,724,1,307.95


In [0]:
# define windows for each metric
r_w = Window.orderBy(col('recency').desc())  # desc so longer recency = lower score
f_w = Window.orderBy(col('frequency'))
m_w = Window.orderBy(col('monetary'))

# rfm score values
rfm = rfm.withColumn('recency_score', ntile(5).over(r_w)) \
         .withColumn('frequency_score', ntile(5).over(f_w)) \
         .withColumn('monetary_score', ntile(5).over(m_w))

display(rfm.head(5))

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,recency,frequency,monetary,recency_score,frequency_score,monetary_score
17399,541,1,-25111.09,1,1,1
12918,627,3,-10953.5,1,2,1
15849,597,1,-5876.34,1,1,1
15760,631,5,-5795.87,1,3,1
16981,541,1,-4620.86,1,1,1


In [0]:
# rfm score
rfm = rfm.withColumn('rfm_score', concat(col('recency_score'), col('frequency_score'), col('monetary_score')))

display(rfm.head(5))

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,recency,frequency,monetary,recency_score,frequency_score,monetary_score,rfm_score
17399,541,1,-25111.09,1,1,1,111
12918,627,3,-10953.5,1,2,1,121
15849,597,1,-5876.34,1,1,1,111
15760,631,5,-5795.87,1,3,1,131
16981,541,1,-4620.86,1,1,1,111


In [0]:
# customer segmentation

rfm = rfm.withColumn('segment',
    when(col('rfm_score').rlike('^[1-2][1-2]'), 'Hibernating')
    .when(col('rfm_score').rlike('^[1-2][3-4]'), 'At Risk')
    .when(col('rfm_score').rlike('^[1-2]5'), 'Can\'t Lose')
    .when(col('rfm_score').rlike('^3[1-2]'), 'About to Sleep')
    .when(col('rfm_score').rlike('^33'), 'Need Attention')
    .when(col('rfm_score').rlike('^[3-4][4-5]'), 'Loyal Customers')
    .when(col('rfm_score').rlike('^41'), 'Promising')
    .when(col('rfm_score').rlike('^51'), 'New Customers')
    .when(col('rfm_score').rlike('^[4-5][2-3]'), 'Potential Loyalists')
    .when(col('rfm_score').rlike('^5[4-5]'), 'Champions')
)

display(rfm.head(5))
                                       

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,recency,frequency,monetary,recency_score,frequency_score,monetary_score,rfm_score,segment
17399,541,1,-25111.09,1,1,1,111,Hibernating
12918,627,3,-10953.5,1,2,1,121,Hibernating
15849,597,1,-5876.34,1,1,1,111,Hibernating
15760,631,5,-5795.87,1,3,1,131,At Risk
16981,541,1,-4620.86,1,1,1,111,Hibernating


In [0]:
rfm_stat = rfm.groupBy('segment').agg(countDistinct('customer_id').alias('customer_count'))
display(rfm_stat)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


segment,customer_count
Hibernating,1563
At Risk,745
Can't Lose,70
Need Attention,266
Loyal Customers,1220
Potential Loyalists,842
About to Sleep,375
Champions,861
